# Step 7 — Visualization & GeoJSON Export

Reconstructs one flight using all three methods and exports to geojson.io.

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6

## Cell 1 — Setup: load model and flight list

In [ ]:
import sys, os
from pathlib import Path
import torch
import pandas as pd
import numpy as np
from pyproj import Geod
from scipy.interpolate import interp1d

# ── Must chdir to noel_part BEFORE importing reconstruction ───────────────────
NOEL_DIR = Path("../noel_part").resolve()
os.chdir(NOEL_DIR)
sys.path.insert(0, str(NOEL_DIR))

print(f"Working dir: {os.getcwd()}")

BASE_DIR  = Path(".")
CLEAN_DIR = BASE_DIR / "cleaned_data_final"
RECON_DIR = BASE_DIR / "final_reconstructions"
OUT_DIR   = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

geod = Geod(ellps="WGS84")

from reconstruction import (TrajectoryBiGRU, FEATURE_COLS, TARGET_COLS,
                             SEQUENCE_LEN, reconstruct_gap_baseline,
                             reconstruct_gap_kalman, reconstruct_gap,
                             compute_features)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TrajectoryBiGRU(len(FEATURE_COLS), 128, 2, len(TARGET_COLS), 0.3).to(device)
model.load_state_dict(torch.load("models/BiGRU.pth", map_location=device))
model.eval()

flights = [f for s in sorted(CLEAN_DIR.iterdir()) if s.is_dir()
           for f in sorted(s.iterdir()) if f.is_dir()]

print(f"Model loaded on {device}")
print(f"Flights available: {len(flights)}")

## Cell 2 — Select a specific flight

We use a known Ireland→New York transatlantic flight for a clean visualization.

In [ ]:
# ── Hardcoded Ireland → New York flight ───────────────────────────────────────
# Change FLIGHT_NAME to visualize a different flight
FLIGHT_NAME = "20230709_a67811_122827_132249"

# Find which step folder contains this flight
STEP_NAME = None
for s in CLEAN_DIR.iterdir():
    if s.is_dir() and (s / FLIGHT_NAME).exists():
        STEP_NAME = s.name
        break

if STEP_NAME is None:
    raise FileNotFoundError(f"Flight {FLIGHT_NAME} not found in {CLEAN_DIR}")

FLIGHT_DIR = CLEAN_DIR / STEP_NAME / FLIGHT_NAME
print(f"Flight : {FLIGHT_NAME}")
print(f"Step   : {STEP_NAME}")

# ── Load data ─────────────────────────────────────────────────────────────────
def _find(fd, names):
    return next((fd / n for n in names if (fd / n).exists()), None)

_ap  = _find(FLIGHT_DIR, ["adsc_clean.parquet",        "adsc.parquet"])
_bp  = _find(FLIGHT_DIR, ["adsb_before_clean.parquet", "adsb_before.parquet"])
_afp = _find(FLIGHT_DIR, ["adsb_after_clean.parquet",  "adsb_after.parquet"])

adsc = pd.read_parquet(str(_ap)).dropna(subset=["latitude","longitude"])
bef  = pd.read_parquet(str(_bp)).dropna(subset=["latitude","longitude"])
aft  = pd.read_parquet(str(_afp)).dropna(subset=["latitude","longitude"])

for _df in [adsc, bef, aft]:
    _df["timestamp"] = pd.to_datetime(_df["timestamp"], utc=True).dt.tz_localize(None)
    _df.sort_values("timestamp", inplace=True)
    _df.reset_index(drop=True, inplace=True)

# ── Gap boundaries ────────────────────────────────────────────────────────────
t_gap_start = bef["timestamp"].iloc[-1]
t_gap_end   = aft["timestamp"].iloc[0]
gap_min     = (t_gap_end - t_gap_start).total_seconds() / 60

# Trim ADS-B to 30 min around the gap boundary
bef_trim = bef[bef["timestamp"] >= t_gap_start - pd.Timedelta(minutes=30)].reset_index(drop=True)
aft_trim = aft[aft["timestamp"] <= t_gap_end   + pd.Timedelta(minutes=30)].reset_index(drop=True)

# ADS-C points strictly inside the gap
adsc_gap = adsc[
    (adsc["timestamp"] >= t_gap_start) &
    (adsc["timestamp"] <= t_gap_end)
].reset_index(drop=True)

print(f"\nADS-B before : {len(bef):>4} pts  (trimmed to {len(bef_trim)})")
print(f"ADS-C        : {len(adsc):>4} pts  ({len(adsc_gap)} inside gap)")
print(f"ADS-B after  : {len(aft):>4} pts  (trimmed to {len(aft_trim)})")
print(f"Gap          : {gap_min:.1f} min")
print(f"From         : lat={bef_trim['latitude'].iloc[-1]:.2f}  lon={bef_trim['longitude'].iloc[-1]:.2f}")
print(f"To           : lat={aft_trim['latitude'].iloc[0]:.2f}  lon={aft_trim['longitude'].iloc[0]:.2f}")

## Cell 3 — Reconstruct the gap with all three methods

All three methods use only ADS-B before and after — ADS-C is never seen during reconstruction.

In [ ]:
# ── Baseline: correct constant-speed great-circle interpolation ───────────────
def reconstruct_forward_only(before_df, after_df, dt=15.0):
    """
    Great-circle interpolation at constant speed between gap endpoints.
    No blending — single arc from before endpoint to after endpoint.
    """
    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    n_steps = max(1, int(round((first_time - last_time).total_seconds() / dt)))

    lat0 = float(before_df["latitude"].iloc[-1]);  lon0 = float(before_df["longitude"].iloc[-1])
    alt0 = float(before_df["altitude"].iloc[-1])
    lat1 = float(after_df["latitude"].iloc[0]);    lon1 = float(after_df["longitude"].iloc[0])
    alt1 = float(after_df["altitude"].iloc[0])

    pts  = geod.npts(lon0, lat0, lon1, lat1, n_steps)
    lats = np.array([p[1] for p in pts])
    lons = np.array([p[0] for p in pts])
    alts = np.linspace(alt0, alt1, n_steps)

    timestamps = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    result = pd.DataFrame({"latitude": lats, "longitude": lons, "altitude": alts})
    result["timestamp"] = timestamps; result["interpolated"] = True
    return result

# ── Retime helper: keep shape, fix timing to constant speed ───────────────────
def retime_to_constant_speed(recon_df, before_df, after_df, dt=15.0):
    """
    Keep the spatial path of the reconstruction but reassign timestamps
    so the aircraft travels at constant speed matching the gap endpoints.
    This fixes the blending slow-down problem in Kalman and BiGRU.
    """
    lats = recon_df["latitude"].values
    lons = recon_df["longitude"].values
    alts = recon_df["altitude"].values if "altitude" in recon_df.columns else np.zeros(len(recon_df))

    # Cumulative arc length along the reconstructed path
    cum_dist = np.zeros(len(lats))
    for i in range(1, len(lats)):
        _, _, d = geod.inv(float(lons[i-1]), float(lats[i-1]),
                           float(lons[i]),   float(lats[i]))
        cum_dist[i] = cum_dist[i-1] + abs(d)

    total_dist = cum_dist[-1]
    if total_dist == 0:
        return recon_df

    last_time  = before_df["timestamp"].iloc[-1]
    first_time = after_df["timestamp"].iloc[0]
    total_sec  = (first_time - last_time).total_seconds()
    n_steps    = max(1, int(round(total_sec / dt)))

    # Constant-speed time fractions → target arc-length fractions
    time_fracs   = np.array([dt*(i+1)/total_sec for i in range(n_steps)])
    target_fracs = np.clip(time_fracs, 0.0, 1.0)
    old_fracs    = cum_dist / total_dist

    f_la  = interp1d(old_fracs, lats, bounds_error=False, fill_value=(lats[0], lats[-1]))
    f_lo  = interp1d(old_fracs, lons, bounds_error=False, fill_value=(lons[0], lons[-1]))
    f_alt = interp1d(old_fracs, alts, bounds_error=False, fill_value=(alts[0], alts[-1]))

    new_ts = [last_time + pd.Timedelta(seconds=dt*(i+1)) for i in range(n_steps)]
    result = pd.DataFrame({
        "latitude":     f_la(target_fracs),
        "longitude":    f_lo(target_fracs),
        "altitude":     f_alt(target_fracs),
        "timestamp":    new_ts,
        "interpolated": True,
    })
    return result

# ── Run all three methods ─────────────────────────────────────────────────────
print("Reconstructing gap with all three methods...")

recon_base   = reconstruct_forward_only(bef_trim, aft_trim)
recon_kalman = compute_features(reconstruct_gap_kalman(bef_trim, aft_trim))
recon_bigru  = compute_features(reconstruct_gap(
    model, bef_trim, aft_trim, FEATURE_COLS, TARGET_COLS, SEQUENCE_LEN, device))

# Retime Kalman and BiGRU to correct constant speed
recon_kalman_rt = retime_to_constant_speed(recon_kalman, bef_trim, aft_trim)
recon_bigru_rt  = retime_to_constant_speed(recon_bigru,  bef_trim, aft_trim)

print(f"Baseline      : {len(recon_base)} steps")
print(f"Kalman        : {len(recon_kalman_rt)} steps (retimed)")
print(f"BiGRU         : {len(recon_bigru_rt)} steps (retimed)")

## Cell 4 — Measure error against ADS-C ground truth

ADS-C was never used during reconstruction — it is now revealed to measure error.

In [ ]:
def error_km(recon_df, truth_df):
    """
    Mean closest-point distance (km) between reconstruction and ADS-C truth.
    Uses nearest-neighbour matching so timestamp misalignment doesn't matter.
    """
    if len(truth_df) == 0 or len(recon_df) == 0:
        return float("nan")

    def haversine_m(lat1, lon1, lat2, lon2):
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat, dlon = lat2-lat1, lon2-lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
        return 2 * 6_371_000 * np.arcsin(np.sqrt(a))

    errs = []
    for _, row in truth_df.iterrows():
        dists = haversine_m(
            row["latitude"], row["longitude"],
            recon_df["latitude"].values, recon_df["longitude"].values,
        )
        errs.append(dists.min())
    return np.mean(errs) / 1000

base_err   = error_km(recon_base,       adsc_gap)
kalman_err = error_km(recon_kalman_rt,  adsc_gap)
bigru_err  = error_km(recon_bigru_rt,   adsc_gap)

print(f"ADS-C waypoints used for evaluation: {len(adsc_gap)}")
print()
print(f"{'Method':<20} {'Mean error (km)':>16}")
print("-" * 38)
print(f"  {'Baseline':<18} {base_err:>14.1f} km")
print(f"  {'Kalman filter':<18} {kalman_err:>14.1f} km")
print(f"  {'BiGRU model':<18} {bigru_err:>14.1f} km")
print()

best = min([("Baseline", base_err), ("Kalman", kalman_err), ("BiGRU", bigru_err)],
           key=lambda x: x[1])
print(f"Best method: {best[0]} ({best[1]:.1f} km)")

## Cell 5 — Export GeoJSON for geojson.io

Exports all three methods + ADS-C ground truth as a single GeoJSON file.
Open at https://geojson.io

In [ ]:
import json

def _line(df, lat_col, lon_col, props):
    coords = [[float(lo), float(la)] for la, lo in zip(df[lat_col], df[lon_col])
              if np.isfinite(float(la)) and np.isfinite(float(lo))]
    if len(coords) < 2:
        return None
    return {"type": "Feature", "properties": props,
            "geometry": {"type": "LineString", "coordinates": coords}}

features = []

# ADS-B boundary context (grey)
features.append(_line(bef_trim, "latitude", "longitude",
    {"label": "ADS-B before (context)",
     "stroke": "#888888", "stroke-width": 2, "stroke-opacity": 0.8}))
features.append(_line(aft_trim, "latitude", "longitude",
    {"label": "ADS-B after (context)",
     "stroke": "#888888", "stroke-width": 2, "stroke-opacity": 0.8}))

# ADS-C ground truth (yellow) — NOT used in reconstruction
features.append(_line(adsc_gap, "latitude", "longitude",
    {"label": f"ADS-C ground truth ({len(adsc_gap)} waypoints)",
     "stroke": "#FFC107", "stroke-width": 4, "stroke-opacity": 1.0}))

# Three independent reconstructions — none saw ADS-C
features.append(_line(recon_base, "latitude", "longitude",
    {"label": f"Baseline (great-circle) — {base_err:.1f} km",
     "stroke": "#F44336", "stroke-width": 2}))
features.append(_line(recon_kalman_rt, "latitude", "longitude",
    {"label": f"Kalman filter — {kalman_err:.1f} km",
     "stroke": "#00BCD4", "stroke-width": 3}))
features.append(_line(recon_bigru_rt, "latitude", "longitude",
    {"label": f"BiGRU model — {bigru_err:.1f} km",
     "stroke": "#4CAF50", "stroke-width": 3}))

features = [f for f in features if f]
fc  = {"type": "FeatureCollection", "features": features}
out = OUT_DIR / f"{FLIGHT_NAME}_final_comparison.geojson"
out.write_text(json.dumps(fc, indent=2), encoding="utf-8")

print(f"GeoJSON saved → {out}")
print("Open at https://geojson.io")
print()
print("Legend:")
print("  Grey   = ADS-B track (boundary context, 30 min around gap)")
print("  Yellow = ADS-C ground truth (held-out, never used in reconstruction)")
print(f"  Red    = Baseline (great-circle)   — {base_err:.1f} km error")
print(f"  Cyan   = Kalman filter              — {kalman_err:.1f} km error")
print(f"  Green  = BiGRU model                — {bigru_err:.1f} km error")

## Cell 6 — (Optional) Find the best North Atlantic flight to visualize

In [ ]:
# ── Find London → New York flights ────────────────────────────────────────────
candidates = []
for fd in flights:
    try:
        b = pd.read_parquet(fd / "adsb_before.parquet")
        a = pd.read_parquet(fd / "adsb_after.parquet")
        bef_lat = b["latitude"].mean(); bef_lon = b["longitude"].mean()
        aft_lat = a["latitude"].mean(); aft_lon = a["longitude"].mean()
        gap = (a["timestamp"].min() - b["timestamp"].max()).total_seconds() / 60
        # Before near UK/Ireland, after near US East Coast
        if (50 < bef_lat < 58 and -10 < bef_lon < 5 and
            38 < aft_lat < 45 and -75 < aft_lon < -60 and 60 < gap < 300):
            candidates.append({
                "name": fd.name, "gap_min": round(gap,1),
                "bef_lat": round(bef_lat,1), "bef_lon": round(bef_lon,1),
                "aft_lat": round(aft_lat,1), "aft_lon": round(aft_lon,1),
            })
    except: continue

df_cand = pd.DataFrame(candidates).sort_values("gap_min")
print(f"London→NY flights found: {len(df_cand)}")
print("\nTo use a different flight, change FLIGHT_NAME in Cell 2 to one of:")
display(df_cand.head(10))